# Elastic Rec Engine Demo

## Section 1 — Create Elastic Serverless Environment

In [ ]:
%%bash
echo "--- Initializing ---"
terraform -chdir=terraform init -upgrade -input=false > /dev/null && echo "Done."
echo "--- Applying Changes ---"
terraform -chdir=terraform apply -auto-approve > /dev/null && echo "Done."

echo "--- Exporting Environment Variables ---"
cat > .env << EOF
ELASTIC_CLOUD_API_KEY=$(terraform -chdir=terraform output -raw elastic_cloud_api_key)
ELASTIC_CLOUD_ID=$(terraform -chdir=terraform output -raw elastic_cloud_id)
HF_TOKEN=$(terraform -chdir=terraform output -raw hf_token)
EOF
echo "Done."

## Section 2 — Ingest Data

In [ ]:
import os
from dotenv import load_dotenv
from elasticsearch import Elasticsearch
from rec_engine.ingest import load_products, augment, bulk_index
from rec_engine.helpers import print_source

load_dotenv()

es = Elasticsearch(
    cloud_id=os.environ["ELASTIC_CLOUD_ID"],
    api_key=os.environ["ELASTIC_CLOUD_API_KEY"]
)

INDEX_NAME             = "products"
EMBEDDING_INFERENCE_ID = ".jina-embeddings-v5-text-small"
RERANKER_INFERENCE_ID  = ".jina-reranker-v3"

products = augment(load_products(), es, EMBEDDING_INFERENCE_ID)
source = products.sample(1, random_state=19).iloc[0]
print_source(source)

bulk_index(es, INDEX_NAME, products)


## Section 3 — The Before: Two Round Trips

In [ ]:
from rec_engine.helpers import print_recommendations

doc = es.get(index=INDEX_NAME, id=source["product_id"], _source_includes=["embedding"])
stored_vector = doc["_source"]["embedding"]

print(f"Request 1 — GET (vector fetch)")
print(f"  Dims: {len(stored_vector)}")
print(f"  Payload: ~{len(stored_vector) * 4 / 1024:.1f} KB returned to client")

results_before = es.search(
    index=INDEX_NAME,
    body={
        "knn": {
            "field":          "embedding",
            "query_vector":   stored_vector,       # vector sent back over the wire
            "k":              10,
            "num_candidates": 50,
            "filter": {"bool": {
                "must":     [{"term": {"in_stock": True}}],
                "must_not": [{"term": {"sku": source["product_id"]}}]
            }}
        },
        "_source": ["sku", "title", "brand", "is_sponsored"]
    }
)

print(f"\nRequest 2 — KNN search")
print(f"  Payload: ~{len(stored_vector) * 4 / 1024:.1f} KB sent to cluster")

print_recommendations(results_before, label="Before — two requests, raw KNN")

## Section 4 — The After: `query_vector_builder.lookup`

In [ ]:
results_after = es.search(
    index=INDEX_NAME,
    body={
        "knn": {
            "field": "embedding",
            "query_vector_builder": {
                "lookup": {
                    "index": INDEX_NAME,
                    "id":    source["product_id"],
                    "path":  "embedding"
                }
            },
            "k":              10,
            "num_candidates": 50,
            "filter": {"bool": {
                "must":     [{"term": {"in_stock": True}}],
                "must_not": [{"term": {"sku": source["product_id"]}}]
            }}
        },
        "_source": ["sku", "title", "brand", "is_sponsored"]
    }
)

print(f"Request count: 1")
print(f"Client payload: 0 KB (no vector sent)")

print_recommendations(results_after, label="After — single request, lookup")

ids_before = [h["_id"] for h in results_before["hits"]["hits"]]
ids_after  = [h["_id"] for h in results_after["hits"]["hits"]]
print(f"Results identical: {ids_before == ids_after}")

## Section 5 — Full Pipeline: lookup + Rerank + Sponsorship Boost

### Without Sponsor Boosting

In [ ]:
results_reranked = es.search(
    index=INDEX_NAME,
    body={
        "retriever": {
            "text_similarity_reranker": {
                "retriever": {
                    "knn": {
                        "field": "embedding",
                        "query_vector_builder": {
                            "lookup": {
                                "index": INDEX_NAME,
                                "id":    source["product_id"],
                                "path":  "embedding"
                            }
                        },
                        "k":              10,
                        "num_candidates": 50,
                        "filter": {"bool": {
                            "must":     [{"term": {"in_stock": True}}],
                            "must_not": [{"term": {"sku": source["product_id"]}}]
                        }}
                    }
                },
                "field":            "product_text",
                "inference_id":     RERANKER_INFERENCE_ID,
                "inference_text":   source["product_text"],
                "rank_window_size": 10
            }
        },
        "_source": ["sku", "title", "brand", "is_sponsored"]
    }
)

print_recommendations(results_reranked, label="After rerank — before boost")


### With Sponsor Boosting

In [ ]:
results_full = es.search(
    index=INDEX_NAME,
    body={
        "retriever": {
            "rescorer": {
                "retriever": {
                    "text_similarity_reranker": {
                        "retriever": {
                            "knn": {
                                "field": "embedding",
                                "query_vector_builder": {
                                    "lookup": {
                                        "index": INDEX_NAME,
                                        "id":    source["product_id"],
                                        "path":  "embedding"
                                    }
                                },
                                "k":              10,
                                "num_candidates": 50,
                                "filter": {"bool": {
                                    "must":     [{"term": {"in_stock": True}}],
                                    "must_not": [{"term": {"sku": source["product_id"]}}]
                                }}
                            }
                        },
                        "field":            "product_text",
                        "inference_id":     RERANKER_INFERENCE_ID,
                        "inference_text":   source["product_text"],
                        "rank_window_size": 10
                    }
                },
                "rescore": {
                    "window_size": 10,
                    "query": {
                        "rescore_query": {
                            "script_score": {
                                "query": {"match_all": {}},
                                "script": {
                                    "source": "return doc['is_sponsored'].value ? 1.0 : 0.0;",
                                }
                            }
                        },
                        "rescore_query_weight": 0.15
                    }
                }
            }
        },
        "_source": ["sku", "title", "brand", "is_sponsored"]
    }
)

print_recommendations(results_full, label="Full pipeline — lookup + rerank + boost")

## Section 6 — Full Comparison

In [ ]:
from rec_engine.helpers import benchmark, print_benchmark

def run_before():
    doc = es.get(index=INDEX_NAME, id=source["product_id"], _source_includes=["embedding"])
    v = doc["_source"]["embedding"]
    es.search(
        index=INDEX_NAME,
        body={
            "knn": {
                "field":          "embedding",
                "query_vector":   v,
                "k":              10,
                "num_candidates": 50,
                "filter": {"bool": {
                    "must":     [{"term": {"in_stock": True}}],
                    "must_not": [{"term": {"sku": source["product_id"]}}]
                }}
            },
            "_source": ["sku", "title", "brand", "is_sponsored"]
        }
    )

def run_after():
    es.search(
        index=INDEX_NAME,
        body={
            "knn": {
                "field": "embedding",
                "query_vector_builder": {
                    "lookup": {
                        "index": INDEX_NAME,
                        "id":    source["product_id"],
                        "path":  "embedding"
                    }
                },
                "k":              10,
                "num_candidates": 50,
                "filter": {"bool": {
                    "must":     [{"term": {"in_stock": True}}],
                    "must_not": [{"term": {"sku": source["product_id"]}}]
                }}
            },
            "_source": ["sku", "title", "brand", "is_sponsored"]
        }
    )

stats = benchmark(
    {"Two requests (before)": run_before, "lookup (after)": run_after},
    warmup=10,
    iterations=100,
)
print_benchmark(stats, baseline="Two requests (before)")

In [ ]:

from rec_engine.helpers import compare_results

compare_results(
    before=results_before,
    after=results_after,
    full=results_full,
    source_title=source["product_title"]
)

## Section 7 — Destroy Environment

In [ ]:
%%bash
terraform -chdir=terraform destroy -auto-approve > /dev/null && echo "Done."
rm -f .env